# Customer Segmentation & Churn Pattern Analytics in European Banking

## Notebook 07: Streamlit Application Preparation

### Objective

This notebook prepares and validates all components required for the Streamlit banking analytics application.

The workflow includes:

- Loading the final churn-prediction model
- Loading the fitted numerical scaler
- Loading the tuned classification threshold
- Recreating the feature-engineering process for new customers
- Validating feature order and data types
- Testing single-customer churn predictions
- Creating reusable prediction utilities
- Preparing application datasets and supporting files

The final Streamlit application will provide:

- Banking KPI analytics
- Interactive customer segmentation
- Churn-pattern visualisations
- Individual churn prediction
- Customer risk classification
- Model explainability
- Business insights and recommendations

In [1]:
import warnings
warnings.filterwarnings("ignore")

import json
import joblib
import numpy as np
import pandas as pd

from pathlib import Path

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
current_directory = Path.cwd()

if current_directory.name.lower() == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

data_directory = project_root / "data"
processed_directory = data_directory / "processed"
model_directory = project_root / "models"
output_directory = project_root / "outputs"
table_directory = output_directory / "tables"

app_directory = project_root / "app"
app_assets_directory = app_directory / "assets"

app_directory.mkdir(
    parents=True,
    exist_ok=True
)

app_assets_directory.mkdir(
    parents=True,
    exist_ok=True
)

model_path = (
    model_directory /
    "final_smote_gradient_boosting.pkl"
)

scaler_path = (
    model_directory /
    "standard_scaler.pkl"
)

threshold_path = (
    model_directory /
    "final_classification_threshold.json"
)

enriched_dataset_path = (
    data_directory /
    "bank_customer_churn_enriched.csv"
)

feature_list_path = (
    processed_directory /
    "engineered_feature_list.csv"
)

print("Project root       :", project_root)
print("Application folder :", app_directory)
print("Model path         :", model_path)
print("Scaler path        :", scaler_path)
print("Threshold path     :", threshold_path)

Project root       : d:\Customer_Segmentation_Churn_Banking
Application folder : d:\Customer_Segmentation_Churn_Banking\app
Model path         : d:\Customer_Segmentation_Churn_Banking\models\final_smote_gradient_boosting.pkl
Scaler path        : d:\Customer_Segmentation_Churn_Banking\models\standard_scaler.pkl
Threshold path     : d:\Customer_Segmentation_Churn_Banking\models\final_classification_threshold.json


In [3]:
required_artifacts = {
    "Final model": model_path,
    "Numerical scaler": scaler_path,
    "Threshold metadata": threshold_path,
    "Enriched dataset": enriched_dataset_path,
    "Feature list": feature_list_path
}

artifact_verification = []

for artifact_name, artifact_path in required_artifacts.items():
    artifact_verification.append({
        "Artifact": artifact_name,
        "Path": str(artifact_path),
        "Exists": artifact_path.exists()
    })

artifact_verification_df = pd.DataFrame(
    artifact_verification
)

artifact_verification_df

,Artifact,Path,Exists
0,Final model,d:\Customer_Segmentation_Churn_Banking\models\...,True
1,Numerical scaler,d:\Customer_Segmentation_Churn_Banking\models\...,True
2,Threshold metadata,d:\Customer_Segmentation_Churn_Banking\models\...,True
3,Enriched dataset,d:\Customer_Segmentation_Churn_Banking\data\ba...,True
4,Feature list,d:\Customer_Segmentation_Churn_Banking\data\pr...,True


In [4]:
missing_artifacts = [
    artifact_name
    for artifact_name, artifact_path
    in required_artifacts.items()
    if not artifact_path.exists()
]

if missing_artifacts:
    raise FileNotFoundError(
        "The following required application artifacts are missing: "
        f"{missing_artifacts}"
    )

print("Validation passed: All application artifacts are available.")

Validation passed: All application artifacts are available.


In [5]:
model = joblib.load(
    model_path
)

scaler = joblib.load(
    scaler_path
)

with open(
    threshold_path,
    "r",
    encoding="utf-8"
) as file:
    threshold_metadata = json.load(file)

classification_threshold = float(
    threshold_metadata[
        "classification_threshold"
    ]
)

model_feature_names = threshold_metadata[
    "feature_names"
]

print("Final model loaded successfully.")
print("Scaler loaded successfully.")
print(
    f"Classification threshold: "
    f"{classification_threshold:.2f}"
)
print(
    f"Expected model features: "
    f"{len(model_feature_names)}"
)

Final model loaded successfully.
Scaler loaded successfully.
Classification threshold: 0.56
Expected model features: 17


In [6]:
df_app = pd.read_csv(
    enriched_dataset_path
)

print("Application dataset loaded successfully.")
print(f"Dataset shape: {df_app.shape}")
print(
    f"Missing values: "
    f"{df_app.isna().sum().sum():,}"
)

df_app.head()

Application dataset loaded successfully.
Dataset shape: (10000, 23)
Missing values: 0


,CustomerId,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,Credit_Card_Status,...,Exited,Churn_Status,CreditScoreBand,AgeGroup,TenureGroup,BalanceStatus,BalanceSegment,HighValueCustomer,EngagementSegment,EDA_Risk_Segment
0,15634602,619,France,Female,42,2,0.00,1,1,Has Credit Card,...,1,Churned,Fair,41-50,New,Zero Balance,Zero Balance,Standard Value,Moderately Engaged,Higher Observed Risk
1,15647311,608,Spain,Female,41,1,83807.86,1,0,No Credit Card,...,0,Retained,Fair,41-50,New,Positive Balance,Low Balance,Standard Value,Moderately Engaged,Higher Observed Risk
2,15619304,502,France,Female,42,8,159660.80,3,1,Has Credit Card,...,1,Churned,Poor,41-50,Loyal,Positive Balance,Premium Balance,High Value,Moderately Engaged,Higher Observed Risk
3,15701354,699,France,Female,39,1,0.00,2,0,No Credit Card,...,0,Retained,Good,31-40,New,Zero Balance,Zero Balance,Standard Value,Moderately Engaged,Lower Observed Risk
4,15737888,850,Spain,Female,43,2,125510.82,1,1,Has Credit Card,...,0,Retained,Excellent,41-50,New,Positive Balance,High Balance,Standard Value,Moderately Engaged,Higher Observed Risk


In [7]:
saved_feature_list = (
    pd.read_csv(feature_list_path)
    ["Feature"]
    .tolist()
)

feature_order_check = pd.DataFrame({
    "Position": range(
        1,
        len(model_feature_names) + 1
    ),
    "Metadata_Feature": model_feature_names,
    "Saved_Feature": saved_feature_list,
    "Matches": [
        metadata_feature == saved_feature
        for metadata_feature, saved_feature
        in zip(
            model_feature_names,
            saved_feature_list
        )
    ]
})

feature_order_check

,Position,Metadata_Feature,Saved_Feature,Matches
0,1,CreditScore,CreditScore,True
1,2,Age,Age,True
2,3,Tenure,Tenure,True
3,4,Balance,Balance,True
4,5,NumOfProducts,NumOfProducts,True
5,6,HasCrCard,HasCrCard,True
6,7,IsActiveMember,IsActiveMember,True
7,8,EstimatedSalary,EstimatedSalary,True
8,9,Geography_Germany,Geography_Germany,True
9,10,Geography_Spain,Geography_Spain,True


In [8]:
if len(model_feature_names) != len(saved_feature_list):
    raise ValueError(
        "Model metadata and saved feature list "
        "contain different feature counts."
    )

if not feature_order_check["Matches"].all():
    raise ValueError(
        "Model feature order does not match "
        "the saved engineered feature list."
    )

print(
    "Validation passed: Model feature order "
    "is correct."
)

Validation passed: Model feature order is correct.


In [9]:
continuous_features = [
    "CreditScore",
    "Age",
    "Tenure",
    "Balance",
    "EstimatedSalary",
    "BalanceSalaryRatio",
    "ProductsPerYear",
    "AgeTenureRatio",
    "ActiveBalance",
    "CustomerValueScore"
]

binary_features = [
    "HasCrCard",
    "IsActiveMember",
    "Geography_Germany",
    "Geography_Spain",
    "Gender_Male"
]

other_numeric_features = [
    "NumOfProducts",
    "ProductEngagementScore"
]

print("Continuous features:", len(continuous_features))
print("Binary features    :", len(binary_features))
print(
    "Other numeric features:",
    len(other_numeric_features)
)

print(
    "Total:",
    len(continuous_features)
    + len(binary_features)
    + len(other_numeric_features)
)

Continuous features: 10
Binary features    : 5
Other numeric features: 2
Total: 17


In [10]:
def prepare_customer_features(
    credit_score,
    geography,
    gender,
    age,
    tenure,
    balance,
    number_of_products,
    has_credit_card,
    is_active_member,
    estimated_salary
):
    """
    Convert raw customer inputs into the exact 17-feature
    format expected by the trained churn model.
    """

    customer_features = pd.DataFrame({
        "CreditScore": [float(credit_score)],
        "Age": [float(age)],
        "Tenure": [float(tenure)],
        "Balance": [float(balance)],
        "NumOfProducts": [
            int(number_of_products)
        ],
        "HasCrCard": [
            int(has_credit_card)
        ],
        "IsActiveMember": [
            int(is_active_member)
        ],
        "EstimatedSalary": [
            float(estimated_salary)
        ],
        "Geography_Germany": [
            int(geography == "Germany")
        ],
        "Geography_Spain": [
            int(geography == "Spain")
        ],
        "Gender_Male": [
            int(gender == "Male")
        ]
    })

    # Recreate engineered features from Notebook 04
    customer_features[
        "BalanceSalaryRatio"
    ] = (
        customer_features["Balance"]
        / (
            customer_features[
                "EstimatedSalary"
            ] + 1
        )
    ).round(4)

    customer_features[
        "ProductsPerYear"
    ] = (
        customer_features[
            "NumOfProducts"
        ]
        / (
            customer_features["Tenure"] + 1
        )
    ).round(4)

    customer_features[
        "AgeTenureRatio"
    ] = (
        customer_features["Age"]
        / (
            customer_features["Tenure"] + 1
        )
    ).round(4)

    customer_features[
        "ActiveBalance"
    ] = (
        customer_features["Balance"]
        * customer_features[
            "IsActiveMember"
        ]
    )

    customer_features[
        "ProductEngagementScore"
    ] = (
        customer_features[
            "NumOfProducts"
        ]
        * (
            customer_features[
                "IsActiveMember"
            ] + 1
        )
    )

    customer_features[
        "CustomerValueScore"
    ] = (
        (
            customer_features["Balance"]
            / 1000
        )
        + (
            customer_features[
                "NumOfProducts"
            ] * 10
        )
        + (
            customer_features[
                "IsActiveMember"
            ] * 20
        )
    ).round(2)

    # Enforce exact model feature order
    customer_features = customer_features[
        model_feature_names
    ].copy()

    # Scale only the continuous columns
    customer_features[
        continuous_features
    ] = scaler.transform(
        customer_features[
            continuous_features
        ]
    )

    return customer_features


print(
    "Single-customer preprocessing "
    "function created successfully."
)

Single-customer preprocessing function created successfully.


In [11]:
def classify_risk_level(
    probability
):
    if probability < 0.30:
        return "Low Risk"

    if probability < classification_threshold:
        return "Medium Risk"

    if probability < 0.75:
        return "High Risk"

    return "Critical Risk"

In [12]:
def predict_customer_churn(
    credit_score,
    geography,
    gender,
    age,
    tenure,
    balance,
    number_of_products,
    has_credit_card,
    is_active_member,
    estimated_salary
):
    """
    Generate the churn probability, binary prediction,
    risk category, and prepared feature row.
    """

    prepared_features = prepare_customer_features(
        credit_score=credit_score,
        geography=geography,
        gender=gender,
        age=age,
        tenure=tenure,
        balance=balance,
        number_of_products=number_of_products,
        has_credit_card=has_credit_card,
        is_active_member=is_active_member,
        estimated_salary=estimated_salary
    )

    churn_probability = float(
        model.predict_proba(
            prepared_features
        )[0, 1]
    )

    predicted_churn = int(
        churn_probability
        >= classification_threshold
    )

    risk_level = classify_risk_level(
        churn_probability
    )

    return {
        "churn_probability": churn_probability,
        "predicted_churn": predicted_churn,
        "prediction_label": (
            "Likely to Churn"
            if predicted_churn == 1
            else "Likely to Stay"
        ),
        "risk_level": risk_level,
        "prepared_features": prepared_features
    }


print(
    "Customer churn prediction "
    "function created successfully."
)

Customer churn prediction function created successfully.


In [13]:
sample_prediction = predict_customer_churn(
    credit_score=650,
    geography="Germany",
    gender="Female",
    age=52,
    tenure=4,
    balance=125000,
    number_of_products=1,
    has_credit_card=1,
    is_active_member=0,
    estimated_salary=85000
)

print("=" * 65)
print("SAMPLE CUSTOMER PREDICTION")
print("=" * 65)

print(
    f"Prediction       : "
    f"{sample_prediction['prediction_label']}"
)

print(
    f"Churn probability: "
    f"{sample_prediction['churn_probability']:.2%}"
)

print(
    f"Risk level       : "
    f"{sample_prediction['risk_level']}"
)

SAMPLE CUSTOMER PREDICTION
Prediction       : Likely to Churn
Churn probability: 94.23%
Risk level       : Critical Risk


In [14]:
prepared_sample = sample_prediction[
    "prepared_features"
]

print("Prepared feature shape:")
print(prepared_sample.shape)

print("\nFeature order:")
for number, feature in enumerate(
    prepared_sample.columns,
    start=1
):
    print(f"{number:02d}. {feature}")

print(
    "\nMissing values:",
    prepared_sample.isna().sum().sum()
)

print(
    "All values finite:",
    np.isfinite(
        prepared_sample.to_numpy()
    ).all()
)

Prepared feature shape:
(1, 17)

Feature order:
01. CreditScore
02. Age
03. Tenure
04. Balance
05. NumOfProducts
06. HasCrCard
07. IsActiveMember
08. EstimatedSalary
09. Geography_Germany
10. Geography_Spain
11. Gender_Male
12. BalanceSalaryRatio
13. ProductsPerYear
14. AgeTenureRatio
15. ActiveBalance
16. ProductEngagementScore
17. CustomerValueScore

Missing values: 0
All values finite: True


In [15]:
app_configuration = {
    "model_name": (
        "SMOTE Gradient Boosting"
    ),
    "classification_threshold": (
        classification_threshold
    ),
    "risk_thresholds": {
        "low_upper_bound": 0.30,
        "medium_upper_bound": (
            classification_threshold
        ),
        "high_upper_bound": 0.75
    },
    "continuous_features": (
        continuous_features
    ),
    "model_feature_names": (
        model_feature_names
    )
}

app_configuration_path = (
    app_directory /
    "app_config.json"
)

with open(
    app_configuration_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        app_configuration,
        file,
        indent=4
    )

print(
    f"Application configuration saved to:\n"
    f"{app_configuration_path}"
)

Application configuration saved to:
d:\Customer_Segmentation_Churn_Banking\app\app_config.json


In [16]:
print("=" * 70)
print("STREAMLIT APPLICATION PREPARATION CHECK")
print("=" * 70)

print(
    f"Model available       : "
    f"{model_path.exists()}"
)

print(
    f"Scaler available      : "
    f"{scaler_path.exists()}"
)

print(
    f"Threshold available   : "
    f"{threshold_path.exists()}"
)

print(
    f"Application data      : "
    f"{enriched_dataset_path.exists()}"
)

print(
    f"Configuration saved   : "
    f"{app_configuration_path.exists()}"
)

print(
    f"Model feature count   : "
    f"{len(model_feature_names)}"
)

print(
    f"Prepared sample shape : "
    f"{prepared_sample.shape}"
)

print(
    f"Selected threshold    : "
    f"{classification_threshold:.2f}"
)

print(
    "\nStreamlit preparation "
    "phase completed successfully."
)

STREAMLIT APPLICATION PREPARATION CHECK
Model available       : True
Scaler available      : True
Threshold available   : True
Application data      : True
Configuration saved   : True
Model feature count   : 17
Prepared sample shape : (1, 17)
Selected threshold    : 0.56

Streamlit preparation phase completed successfully.
